In [1]:
import torch
import pandas as pd
from computer_ontology.config import*
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from computer_ontology.featurizer import get_mordred
from torchmetrics.classification import MultilabelF1Score, MultilabelAUROC, MultilabelPrecision, MultilabelRecall

In [2]:
# fetching the datasets
train = pd.read_csv(alldesc_train_path_2025)
test = pd.read_csv(alldesc_test_path_2025)

In [3]:
train_x = train['IsomericSMILES']
train_y = train.drop(columns=['CID', 'IsomericSMILES', 'Descriptors'], axis=1, inplace = False)

test_x = test['IsomericSMILES']
test_y = test.drop(columns=['CID', 'IsomericSMILES', 'Descriptors'], axis=1, inplace = False)

In [4]:
top_classes = (
    train_y.sum()
    .sort_values(ascending=False)
    .head(16)
    .index
)

In [5]:
train_y_reduced = train_y[top_classes]

mask = train_y_reduced.sum(axis=1) > 0
train_x_reduced = train_x[mask]
train_y_reduced = train_y_reduced[mask]

In [6]:
train_y_reduced

,fruity,green,floral,herbal,woody,fat,spicy,citrus,wax,balsam,rose,earth,sulfur,tropical,ether,apple
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
8,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5325,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0
5326,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0
5327,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0
5328,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
test_y_reduced = test_y[top_classes]

mask = test_y_reduced.sum(axis=1) > 0
test_x_reduced = test_x[mask]
test_y_reduced = test_y_reduced[mask]

In [8]:
rf_params_2025

OrderedDict([('random_state', 0),
             ('bootstrap', False),
             ('max_depth', 90),
             ('max_features', 'log2'),
             ('min_samples_leaf', 1),
             ('min_samples_split', 2),
             ('n_estimators', 1800)])

In [ ]:
train_x = get_mordred(train_x)
test_x = get_mordred(test_x)

train_x = train_x[selected_features_2025]
test_x = test_x[selected_features_2025]

train_x.shape

scaler = MinMaxScaler()

scaler.fit(train_x)

train_x = pd.DataFrame(data=scaler.transform(train_x), columns=train_x.columns)
test_x = pd.DataFrame(data=scaler.transform(test_x), columns=test_x.columns)

clf = RandomForestClassifier(**rf_params_2025, n_jobs=1)

clf.fit(train_x, train_y)

y_hat = clf.predict(test_x)

f1score_macro = MultilabelF1Score(num_labels=len(train_y.columns), average="macro")(torch.tensor(y_hat, dtype=torch.float), torch.tensor(test_y.values, dtype=torch.float))
auroc_macro = MultilabelAUROC(num_labels=len(train_y.columns), average="macro")(torch.tensor(y_hat, dtype=torch.float), torch.tensor(test_y.values, dtype=torch.long))
precision_macro = MultilabelPrecision(num_labels=len(train_y.columns), average="macro")(torch.tensor(y_hat, dtype=torch.float), torch.tensor(test_y.values, dtype=torch.long))
recall_macro = MultilabelRecall(num_labels=len(train_y.columns), average="macro")(torch.tensor(y_hat, dtype=torch.float), torch.tensor(test_y.values, dtype=torch.long))

print(f"F1 Score Macro: {f1score_macro}")
print(f"AUROC Macro: {auroc_macro}")
print(f"Precision Macro: {precision_macro}")
print(f"Recall Macro: {recall_macro}")

100%|██████████| 1380/1380 [02:10<00:00, 10.59it/s]
